# 🔄 Basic Agent Workflows with Azure AI Foundry (Python)

## 📋 Workflow Orchestration Tutorial

This notebook introduces the powerful **Workflow Builder** capabilities of the Microsoft Agent Framework. Learn how to create sophisticated, multi-step agent workflows that can handle complex business processes and coordinate multiple AI operations seamlessly.

## 🎯 Learning Objectives

### 🏗️ **Workflow Architecture**
- **Workflow Builder**: Design and orchestrate complex multi-step processes
- **Event-Driven Execution**: Handle workflow events and state transitions
- **Visual Workflow Design**: Create and visualize workflow structures
- **Azure AI Foundry Integration**: Leverage Foundry model deployments within workflow contexts

### 🔄 **Process Orchestration**
- **Sequential Operations**: Chain multiple agent tasks in logical order
- **Conditional Logic**: Implement decision points and branching workflows
- **Error Handling**: Robust error recovery and workflow resilience
- **State Management**: Track and manage workflow execution state

### 📊 **Enterprise Workflow Patterns**
- **Business Process Automation**: Automate complex organizational workflows
- **Multi-Agent Coordination**: Coordinate multiple specialized agents
- **Scalable Execution**: Design workflows for enterprise-scale operations
- **Monitoring & Observability**: Track workflow performance and outcomes

## ⚙️ Prerequisites & Setup

### 📦 **Required Dependencies**

Install the Agent Framework with Azure AI Foundry support:

```bash
pip install agent-framework-azure-ai -U --pre
```

### 🔑 **Azure AI Foundry Configuration**

**Environment Setup (.env file):**
```env
FOUNDRY_PROJECT_ENDPOINT=your_foundry_project_endpoint
FOUNDRY_MODEL=your_foundry_model_deployment
```

**Authentication Setup:**
```bash
az login
az account set --subscription "your-subscription-id"
```

### 🏢 **Enterprise Use Cases**

**Business Process Examples:**
- **Customer Onboarding**: Multi-step verification and setup workflows
- **Content Pipeline**: Automated content creation, review, and publishing
- **Data Processing**: ETL workflows with AI-powered transformation
- **Quality Assurance**: Automated testing and validation processes

**Workflow Benefits:**
- 🎯 **Reliability**: Deterministic execution with error recovery
- 📈 **Scalability**: Handle high-volume process automation
- 🔍 **Observability**: Complete audit trails and monitoring
- 🔧 **Maintainability**: Visual design and modular components

## 🎨 Workflow Design Patterns

### Basic Workflow Structure
```mermaid
graph TD
    A[Start] --> B[Agent Task 1]
    B --> C{Decision Point}
    C -->|Success| D[Agent Task 2]
    C -->|Failure| E[Error Handler]
    D --> F[End]
    E --> F
```

**Key Components:**
- **WorkflowBuilder**: Main orchestration engine
- **WorkflowEvent**: Event handling and communication
- **WorkflowViz**: Visual workflow representation and debugging
- **FoundryChatClient**: Azure AI Foundry-backed model client

Let's build your first intelligent workflow! 🚀

In [ ]:
####### Linux / MacOS Setup Instructions #######

####### Linux Instructions #######
# ! sudo apt update 
# ! sudo apt install graphviz -y

####### macOS Instructions #######
# ! brew install graphviz

In [ ]:
# ! pip install -r ../../../Installation/requirements.txt -U
# 

In [ ]:
# 🔄 Import Workflow and Agent Framework Components
# Core components for building sophisticated agent workflows

from azure.identity import AzureCliCredential
from agent_framework.foundry import FoundryChatClient  # 🤖 Azure AI Foundry client integration
from agent_framework import AgentResponse, WorkflowBuilder, WorkflowEvent, WorkflowViz  # 🏗️ Workflow orchestration tools

In [ ]:
# 📦 Import Environment and System Utilities
# Essential libraries for configuration and environment management

from typing import cast
from dotenv import load_dotenv # 📁 Secure configuration loading

In [ ]:
# 🔧 Initialize Environment Configuration
# Load Azure AI Foundry project endpoint and model deployment from .env file
load_dotenv()

In [ ]:
# 🔗 Initialize Azure AI Foundry Chat Client for Workflow Operations
# Create the AI client that will power agents within our workflow
chat_client = FoundryChatClient(
    credential=AzureCliCredential()
)

In [ ]:
REVIEWER_NAME = "Concierge"
REVIEWER_INSTRUCTIONS = """
    You are an are hotel concierge who has opinions about providing the most local and authentic experiences for travelers.
    The goal is to determine if the front desk travel agent has recommended the best non-touristy experience for a traveler.
    If so, state that it is approved.
    If not, provide insight on how to refine the recommendation without using a specific example. 
    """

In [ ]:
FRONTDESK_NAME = "FrontDesk"
FRONTDESK_INSTRUCTIONS = """
    You are a Front Desk Travel Agent with ten years of experience and are known for brevity as you deal with many customers.
    The goal is to provide the best activities and locations for a traveler to visit.
    Only provide a single recommendation per response.
    You're laser focused on the goal at hand.
    Don't waste time with chit chat.
    Consider suggestions when refining an idea.
    """

In [ ]:
reviewer_agent   = chat_client.as_agent(
        instructions=(
           REVIEWER_INSTRUCTIONS
        ),
        name=REVIEWER_NAME,
    )

front_desk_agent = chat_client.as_agent(
        instructions=(
            FRONTDESK_INSTRUCTIONS
        ),
        name=FRONTDESK_NAME,
    )

In [ ]:
workflow = WorkflowBuilder(start_executor=front_desk_agent).add_edge(front_desk_agent, reviewer_agent).build()

In [ ]:

print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
svg_file = viz.export(format="svg")
print(f"SVG file saved to: {svg_file}")

In [ ]:
class DatabaseEvent(WorkflowEvent): ...

In [ ]:
# Display the exported workflow SVG inline in the notebook

from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")


In [ ]:
events = await workflow.run("I would like to go to Paris.")

outputs = events.get_outputs()
    # The outputs of the workflow are whatever the agents produce. So the outputs are expected to be a list
    # of `AgentResponse` from the agents in the workflow.
outputs = cast(list[AgentResponse], outputs)
for output in outputs:
    print(f"{output.messages[0].author_name}: {output.text}\n")

    # Summarize the final run state (e.g., COMPLETED)
print("Final state:", events.get_final_state())